In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [ ]:
# =============================================================================
# LONDON CONFIGURATION BLOCK
# Source: NTS Data Extract User Guide + Main Table Variables London.csv
# =============================================================================

# ---- Year filter (applied at load time for memory efficiency) ----
survey_year = 2023

# ---- Geographic filter ----
# London code — PSUStatsReg_B01ID 
london_code = 8  

# ---- Trip purpose filter (TripPurpose_B04ID, 8 categories) ----
commute_code_london = 1

# ---- Mode codes (MainMode_B04ID, 13 categories) ----
mode_car_driver  = 3
mode_bicycle     = 2
mode_bus_london  = 7
mode_bus_other   = 8
mode_underground = 10 # Underground + DLR + light rail + tram
mode_other_pt = 13 

# Choice set: car driver, PT (bus + underground), bicycle
pt_codes_london  = [mode_bus_london, mode_bus_other, mode_underground, mode_other_pt]
choice_modes_london = [mode_car_driver] + pt_codes_london + [mode_bicycle]

# ---- Weekday filter (TravelWeekDay_B01ID) ----
weekday_codes_london = [1, 2, 3, 4, 5]   # Mon–Fri

# ---- Holiday filter (TravelDayType_B01ID, from 2008 onwards) ----
# 1 = Normal day, 2+ = bank holiday / special day
normal_day_code = 1

# ---- Socioeconomic missing codes (NTS convention) ----
# -8 = "Don't know", -9 = "Not answered/applicable"
missing_codes = [-8, -9, -10]

# ---- Driving licence (DrivLic_B02ID, 3 categories) ----
# 1 = Full licence, 2 = Provisional, 3 = No licence
full_licence_code = 1

# ---- Income variable (year-specific quintiles for England) ----
income_var_2023 = "HHIncQIS2023Eng_B01ID"  # interview-sample quintiles

# ---- Purpose labels (for EDA bar chart) ----
purpose_labels = {
    1: "Commuting", 2: "Business", 3: "Education", 4: "Escort",
    5: "Shopping", 6: "Other personal", 7: "Leisure", 8: "Other/walk"
}

# --- Distance conversion ---
miles_to_km = 1.609344    # AfstV is recorded in hectometers

# --- Cleaning thresholds --- Same as in Amsterdam dataset
min_trip_dur = 2 # in mins; below 2mins likely data entry error
max_trip_dur = 120 # in mins; 
min_trip_distance = 0.2 # in km; below is unlikely for any of our three modes
max_trip_distance = 50 # in km;

# --- Peak hour definition --- Same as in Amsterdam Dataset
am_peak_start = 6 
am_peak_end   = 9
pm_peak_start = 16
pm_peak_end   = 19

# ---- Mode labels (for EDA charts, all 13 categories) ----
mode_labels_b04 = {
     1: "Walk",  2: "Bicycle",  3: "Car driver",  4: "Car passenger",
     5: "Motorcycle",  6: "Other private",  7: "Bus (London)",
     8: "Other local bus",  9: "Non-local bus/coach",
    10: "Underground/metro/LR", 11: "Surface rail",
    12: "Taxi/minicab", 13: "Other public"
}


In [ ]:
# Shared columns list
shared_columns = [
    "person_id", "trip_id", "trip_purpose", "mode_detailed",
    "travel_time_min", "distance_km", "departure_hour",
    "weekday", "is_holiday", "age_band", "gender",
    "income_quintile", "has_driving_license", "n_cars_household",
    "weight_trip", "weight_person", "city", "is_peak", "chosen_mode",
    "n_transfers", "has_transfer", "n_legs"
]

# ---- Choice set labels (dependent variable) ----
choice_set = ["car", "pt", "bike"]

# ---- Age bands (matching NTS Age_B04ID 9-category scheme) ----
#   Code 1: 0-4, 2: 5-10, 3: 11-16, 4: 17-20, 5: 21-29,
#   6: 30-39, 7: 40-49, 8: 50-59, 9: 60+
age_bins  = [0, 5, 11, 17, 21, 30, 40, 50, 60, 200]
age_labels = [1, 2, 3, 4, 5, 6, 7, 8, 9]

# ---- Income: Amsterdam decile → quintile mapping ----
# ODiN HHGestInkG deciles 1-10 → quintiles 1-5
decile_to_quintile= {
    1: 1, 2: 1,   # Q1 (lowest)
    3: 2, 4: 2,   # Q2
    5: 3, 6: 3,   # Q3
    7: 4, 8: 4,   # Q4
    9: 5, 10: 5,  # Q5 (highest)
}

In [ ]:
# Read ODiN 2023 trip-level data
uk_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\implementation notebooks-Copy1\uk_data_2023.csv") #update the path accordingly to load the data   

In [ ]:
mode_counts_all = (
    uk_data.MainMode_B04ID.value_counts().sort_index().rename(index = mode_labels_b04)
)

mode_pct_all = (mode_counts_all / mode_counts_all.sum() * 100).round(2)
mode_table_all = pd.DataFrame({"count": mode_counts_all, "pct": mode_pct_all})

In [ ]:
mode_table_all

In [ ]:
# Plot the mode share in the UK trips
plot_df = mode_table_all.reset_index()

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data = plot_df, 
    x = 'MainMode_B04ID', 
    y = 'count',
    hue = 'MainMode_B04ID',
    palette = 'viridis',
    legend=False
)

# Percentages on top of each bar
for i, p in enumerate(ax.patches):
    # Get the percentage from the 'pct' column
    percentage = plot_df['pct'].iloc[i]
    
    ax.annotate(f'{percentage:.1f}%', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 9), 
                textcoords='offset points',
                fontsize=11,
                fontweight='bold')

plt.title('UK Transport Mode Share', fontsize = 15, pad = 20)
plt.xlabel('Transport Mode', fontsize = 12)
plt.ylabel('Number of Trips (Count)', fontsize = 12)

# Increase Y-limit to make room for labels
plt.ylim(0, plot_df['count'].max() * 1.15)

plt.grid(axis='y', linestyle='--', alpha=0.5)
sns.despine()

plt.xticks(rotation = 25, ha = "right")
plt.tight_layout()
plt.show()

In [ ]:
# Then we are going to filter to car driver + PT + bicycle only
target_modes_mask = uk_data.MainMode_B04ID.isin(choice_modes_london)
uk_target_modes = uk_data[target_modes_mask].copy()

In [ ]:
# Count and percentage by purpose
purpose_counts = (uk_target_modes.TripPurpose_B04ID.value_counts().sort_index().rename(index=purpose_labels))
purpose_pct = (purpose_counts / purpose_counts.sum() * 100).round(2)
purpose_table = pd.DataFrame({"count": purpose_counts, "pct": purpose_pct})

In [ ]:
# Lets now visualize the mode share in UK
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(purpose_table.index, purpose_table["count"], color="#4C72B0")

for bar, pct in zip(bars, purpose_table["pct"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f"{pct}%", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_title("Trip Purpose Distribution — UK NTS 2023 (Car, PT & Bicycle Trips Only)", fontsize = 12, fontweight = "bold")
ax.set_ylabel("Number of Trips")
ax.set_xlabel("Trip Purpose")
# ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

plt.xticks(rotation = 25, ha = "right")
plt.tight_layout()
plt.show()

In [ ]:
# Collapsed into our 3 choice-set categories
car_n = uk_data.MainMode_B04ID.eq(mode_car_driver).sum()
pt_n = uk_data.MainMode_B04ID.isin(pt_codes_london).sum()
bike_n = uk_data.MainMode_B04ID.eq(mode_bicycle).sum()
other_n = len(uk_data) - car_n - pt_n - bike_n
total = len(uk_data)

In [ ]:
print("Collapsed to choice set (car / PT / bike / other):")
for label, n in [("Car driver", car_n), ("PT (bus+underground)", pt_n),
                  ("Bicycle", bike_n), ("Other modes", other_n)]:
    print(f"  {label:25s}: {n:>7,}  ({100*n/total:.1f}%)")

In [ ]:
# Reuse the apply_filter function from Amsterdam pipeline
transformation_log_london = []

def apply_filter(dataset, mask, description, log = transformation_log_london):
    n_before = len(dataset)
    dataset_r = dataset[mask].copy()
    n_after = len(dataset_r)
    n_dropped = n_before - n_after
    entry = f"{description}: {n_before:,} → {n_after:,} rows (dropped {n_dropped:,})"
    log.append(entry)
    print(entry)
    return dataset_r

london_data = uk_data.copy()

# F1: London only
london_data = apply_filter(
    london_data,
    london_data["PSUStatsReg_B01ID"] == london_code,
    f"F1: London residents (PSUStatsReg_B01ID == {london_code})"
)

In [ ]:
# Temporarily filter to commuting only for this chart
london_commute_temp = london_data[london_data.TripPurpose_B04ID == commute_code_london].copy()

mode_dist_ldn = (
    london_commute_temp.MainMode_B04ID.value_counts().sort_index().rename(index = mode_labels_b04)
)

mode_pct_ldn = (mode_dist_ldn / mode_dist_ldn.sum() * 100).round(2)
mode_table_ldn = pd.DataFrame({"count": mode_dist_ldn, "pct": mode_pct_ldn})

#lets plot the mode distribution in London
fig, ax = plt.subplots(figsize=(11, 5))

bars = ax.bar(mode_table_ldn.index, mode_table_ldn["count"], edgecolor="white", linewidth=0.5)

for bar, pct in zip(bars, mode_table_ldn["pct"]):
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
                f"{pct}%", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_title("Mode Distribution — London Commuting (Pre-Filter)\n",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Number of Trips")
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# F2: Commuting trips only
london_data = apply_filter(
    london_data,
    london_data["TripPurpose_B04ID"] == commute_code_london,
    "F2: Commuting trips only (TripPurpose_B04ID == 1)"
)

# F3: Weekdays only (Mon–Fri)
london_data = apply_filter(
    london_data,
    london_data["TravelWeekDay_B01ID"].isin(weekday_codes_london),
    "F3: Weekdays only (Mon–Fri)"
)

# F4: Exclude public holidays / bank holidays
# A mask for a "pure" working day -> weekday which is not a bank holiday

london_data = apply_filter(
    london_data,
    london_data.TravelWeekDay_B02ID == normal_day_code,
    # normal_day_mask,
    f"F4: Normal days only (TravelDayType_B01ID == {normal_day_code})"
)

# F5: Three modes only — car driver, PT (bus + underground), bicycle
london_data = apply_filter(
    london_data,
    london_data["MainMode_B04ID"].isin(choice_modes_london),
    "F5: Choice set modes only (car driver, PT, bicycle)"
)

# F6: Exclude short walk trips coded as trips
london_data = apply_filter(
    london_data,
    london_data["ShortWalkTrip_B01ID"] != 1,
    "F6: Exclude short walk trips (ShortWalkTrip_B01ID != 1)"
)

# F7: Exclude missing income
london_data = apply_filter(
    london_data,
    ~london_data[income_var_2023].isin(missing_codes),
    f"F7: Exclude missing income ({income_var_2023} not in {missing_codes})"
)

# F8: Exclude missing driving licence
london_data = apply_filter(
    london_data,
    ~london_data["DrivLic_B02ID"].isin(missing_codes),
    "F8: Exclude missing driving licence (DrivLic_B02ID)"
)

# F9: Exclude missing car ownership
london_data = apply_filter(
    london_data,
    ~london_data["NumCarVan"].isin(missing_codes),
    "F9: Exclude missing car ownership (NumCarVan)"
)

In [ ]:
london_clean = london_data.copy()

conditions = [
    london_clean["MainMode_B04ID"] == mode_car_driver,
    london_clean["MainMode_B04ID"].isin(pt_codes_london),
    london_clean["MainMode_B04ID"] == mode_bicycle,
]
labels = ["car", "pt", "bike"]
london_clean["chosen_mode"] = np.select(conditions, labels, default = None)

miles_to_km = 1.609344
london_clean["distance_km"] = london_clean.TripDisIncSW * miles_to_km # Convert distance: miles to km = 1.609344

london_clean["TripStartHours"] = pd.to_numeric(london_clean.TripStartHours, errors='coerce') #We convert the data to numeric so we make the 29 records as NaN
london_clean["TripStartHours"] = london_clean.TripStartHours.fillna(-1).astype(int) # Fill the 29 NaNs with a non-peak placeholder = -1, so we do not lose the records - we flag them. 

In [ ]:
# Insert peak-hour flag
london_clean["is_peak"] = (
    london_clean.TripStartHours.between(am_peak_start, am_peak_end) | 
    london_clean.TripStartHours.between(pm_peak_start, pm_peak_end)
).astype(int)

# Has driving licence 1 = full
london_clean["has_driving_license"] = (london_clean.DrivLic_B02ID == full_licence_code).astype(int)

# Add city column
london_clean["city"] = "London"

In [ ]:
# rename columns inline with Amsterdam dataset
rename_map = {
    "IndividualID": "person_id",
    "TripID": "trip_id",
    "TripPurpose_B04ID": "trip_purpose",
    "MainMode_B04ID": "mode_detailed",
    "TripTravTime": "travel_time_min",
    "TripStartHours": "departure_hour",
    "NumStages" : "n_legs",
    "TravelWeekDay_B01ID": "weekday",
    "TravelDayType_B01ID": "is_holiday",
    "Age_B04ID": "age_band",
    "Sex_B01ID": "gender",
    income_var_2023: "income_quintile",
    "NumCarVan": "n_cars_household",
    "W5": "weight_trip",
    "W3": "weight_person"
}
london_clean = london_clean.rename(columns = rename_map) 

# Select only shared schema columns
london_clean = london_clean[shared_columns].copy()

#We convert the n_cars_households and weight_trip from object to respectively float and int 
london_clean['n_cars_household'] = pd.to_numeric(london_clean.n_cars_household, errors = 'coerce').astype('int64')
london_clean['weight_trip'] = pd.to_numeric(london_clean.weight_trip, errors = 'coerce')

In [ ]:
london_clean.chosen_mode.value_counts()

In [ ]:
london_clean.shape

In [ ]:
#F10 Travel time between: 2 and 120 min
london_clean = apply_filter(
    london_clean,
    london_clean.travel_time_min.between(min_trip_dur, max_trip_dur),
    f"F10: Travel time within {min_trip_dur}–{max_trip_dur} min"
)

#F11 Distance: 0.2–50 km
london_clean = apply_filter(
    london_clean,
    london_clean.distance_km.between(min_trip_distance, max_trip_distance),
    f"F11: Distance {min_trip_distance}–{max_trip_distance} km"
)

In [ ]:
variables  = ["travel_time_min", "distance_km"]
mode_colors = {"car": "#2196F3", "pt": "#FF9800", "bike": "#4CAF50"}
transport_modes = ["car", "pt", "bike"]
mode_labels = {"car": "Car (driver)", "pt": "Public Transport", "bike": "Bicycle"}

fig, axes = plt.subplots(2,1, figsize = (8, 8),sharex = False ,sharey = True)

for ax, var in zip(axes, variables):
    for mode in transport_modes:
        subset = london_clean[london_clean.chosen_mode == mode]
        # Weighted histogram using survey weights
        ax.hist(
            subset[var],
            weights = subset.weight_trip,
            bins = 30,
            alpha = 0.55,
            color = mode_colors[mode],
            label = mode_labels[mode],
            edgecolor = "none"
        )
    ax.set_xlabel("Travel time (minutes)" if var == "travel_time_min" else "Distance (km)", fontsize = 10)
    ax.set_ylabel("Weighted frequency", fontsize = 10)
    ax.set_title("Travel time distribution by mode" if var == "travel_time_min" else "Trip distance distribution by mode",
                 fontsize = 11, fontweight = "bold")
    ax.legend(fontsize = 9)
    ax.grid(axis = "y", alpha = 0.3, linewidth = 0.5)
    ax.spines[["top", "right"]].set_visible(False)

plt.subplots_adjust(hspace = 0.4)
plt.show()

In [ ]:
#1. Melt the data so variables are in a single column for grouping
lnd_melted = london_clean.melt(id_vars = ['chosen_mode'], value_vars = variables)

# 2. Calculate Q1 and Q3 per group
lnd_stats = lnd_melted.groupby(['variable', 'chosen_mode'],observed = False).value.quantile([0.25, 0.75]).unstack()
lnd_stats.columns = ['Q1', 'Q3']

# 3. Define the upper and lower boundaries (fences)
lnd_stats['iqr'] = lnd_stats.Q3 - lnd_stats.Q1 
lnd_stats['lower'] = lnd_stats.Q3 - 1.5 * lnd_stats.iqr
lnd_stats['upper'] = lnd_stats.Q3 + 1.5 * lnd_stats.iqr

# 4. Count outliers by merging stats back to the melted data
lnd_merged = lnd_melted.merge(lnd_stats, on = ['variable', 'chosen_mode'])
lnd_merged['is_outlier'] = (lnd_merged.value < lnd_merged.lower) | (lnd_merged.value > lnd_merged.upper)
n_outliers = lnd_merged.groupby(['variable', 'chosen_mode'], observed = False).is_outlier.sum().reset_index()

# 5. Build final report
lnd_report = lnd_stats.reset_index().merge(n_outliers, on = ['variable', 'chosen_mode'])
lnd_report['mode'] = lnd_report.chosen_mode.map(mode_labels)

In [ ]:
lnd_stats.head(5)

In [ ]:
lnd_report

In [ ]:
# --- Export ---
london_clean.to_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_processed.csv", index=False)
print(f"\n Processed dataset saved to data\processed")